In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

In [4]:
df = pd.read_csv('../data/spam.csv',encoding='latin1')

In [5]:
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [6]:
df.shape

(5728, 2)

In [7]:
df.isna().sum()

text    0
spam    0
dtype: int64

In [8]:
df['spam'].value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

In [9]:
"""
    Clean and preprocess text:
    1. Lowercase
    2. Remove URLs
    3. Remove punctuation
    4. Remove numbers
    5. Remove stopwords
    6. Stemming
"""

'\n    Clean and preprocess text:\n    1. Lowercase\n    2. Remove URLs\n    3. Remove punctuation\n    4. Remove numbers\n    5. Remove stopwords\n    6. Stemming\n'

In [10]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to C:\Users\BHUMIT
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\BHUMIT
[nltk_data]     SINGH\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
stop_words = set(stopwords.words('english'))

In [12]:
import string
import emoji

def rm_emoji(x):
    return emoji.replace_emoji(x, replace='')   
    
def rm_pn(x):
    return x.translate(str.maketrans('','',string.punctuation))

def rm_num(x):
    txt=""
    for i in x:
        if(not i.isdigit()):
            txt+=i
    return txt

def rm_stop(x):
    arr = word_tokenize(x)
    cleaned=[]
    for i in arr:
        if(not i in stop_words):
            cleaned.append(i)
    return " ".join(cleaned)


In [13]:
def clean_text(x):
    """Chain all cleaning steps in order."""
    x = x.lower()
    x = rm_pn(x)        # Step 1: Remove punctuation
    x = rm_num(x)       # Step 2: Remove numbers
    x = rm_stop(x)      # Step 3: Remove stopwords
    x = rm_emoji(x)     # Step 3: Remove emojies
    return x

In [14]:
df['text']=df['text'].apply(clean_text)

In [15]:
df['text'][0]

'subject naturally irresistible corporate identity lt really hard recollect company market full suqgestions information isoverwhelminq good catchy logo stylish statlonery outstanding website make task much easier promise havinq ordered iogo company automaticaily become world ieader isguite ciear without good products effective business organization practicable aim hotat nowadays market promise marketing efforts become much effective list clear benefits creativeness hand made original logos specially done reflect distinctive company image convenience logo stationery provided formats easy use content management system letsyou change website content even structure promptness see logo drafts within three business days affordability marketing break make gaps budget satisfaction guaranteed provide unlimited amount changes extra fees surethat love result collaboration look portfolio interested'

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X = df['text']
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)  

In [18]:
text_pipe = Pipeline([
    ('clean', FunctionTransformer(clean_text)),
    ('vec',CountVectorizer())
])

In [19]:
from sklearn.compose import ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ('text_preprocess_pipeline',text_pipe,"text")
    ]
)

In [20]:
pipe = Pipeline([
    ('vec', CountVectorizer(preprocessor=clean_text)),
    ('model', MultinomialNB())
])

In [21]:
random_cv = RandomizedSearchCV(
    estimator=pipe,
    param_distributions={
        # Chained path to reach CountVectorizer
        'vec__max_features':[5000, 10000, 20000],
        'vec__ngram_range': [(1, 1), (1, 2)],
        
        # Correct step name for MultinomialNB ('model' instead of 'clf')
        'model__alpha': [0.1, 1, 10, 100],
        'model__fit_prior': [True, False],
        'model__force_alpha': [True, False],
    },
    n_iter=50,
    cv=5,
    scoring='accuracy',
    random_state=42,
)


In [ ]:
random_cv.fit(X_train, y_train)

In [23]:
pipe.fit(X_train, y_train )

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vec', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...00266810CCB80>
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None


In [24]:
y_pred = pipe.predict(X_test)

In [25]:
accuracy_score(y_pred,y_test)

0.9883659439450027